<table style="border:0; background:none; width:100%">
<tr>
<td style="vertical-align: middle; width: 60%;">
<b>Pontificia Universidad Javeriana</b><br/>
Facultad de Ingeniería · Departamento de Ingeniería de Sistemas<br/>
<b>Procesamiento de Datos a Gran Escala</b>
</td>
<td style="vertical-align: middle; text-align: right; width: 40%;">
<b>Proyecto de Investigación — Entrega 2</b><br/>
Brecha digital y resultados Saber 11 en Colombia<br/>
<b>Grupo: REST pAPIs</b>
</td>
</tr>
</table>

---

# 01 — Bronze: CSV → Parquet

Convertimos los 4 archivos CSV de la capa Bronze a **Parquet con compresión Snappy**.

**¿Por qué Parquet?**
- **Columnar:** lee solo las columnas que necesitas (clave en ICFES con 51 columnas).
- **Splittable + comprimido:** ~3-5× menos espacio que CSV en disco; ~5-10× más rápido de leer en Spark.
- **Schema embebido:** no tenemos que reinferir cada vez.

**Estrategia:**
- Mantener todo como `string` por ahora (la limpieza/casting va en Silver).
- Sobrescribir si ya existe (modo `overwrite`).
- Verificar conteo de filas antes vs después.

## 1. Setup

In [1]:
import sys, time
sys.path.insert(0, "/home/estudiante/proyecto_datos/scripts")
from common_spark import get_spark, P, HDFS_NAMENODE
from pyspark.sql import functions as F

spark = get_spark(
    "Entrega2-Bronze",
    executor_memory="4g",
    driver_memory="2g",
    cores=2,
)
sc = spark.sparkContext
print("App ID :", sc.applicationId)
print("Spark UI (sustituye `spark-master` por 10.43.97.164):", sc.uiWebUrl)

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


26/05/19 16:32:16 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


26/05/19 16:32:17 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


App ID : app-20260519163217-0019
Spark UI (sustituye `spark-master` por 10.43.97.164): http://spark-master:4041


In [2]:
# Helper para tamaño de cualquier ruta en HDFS
URI = sc._jvm.java.net.URI
Path = sc._jvm.org.apache.hadoop.fs.Path
FileSystem = sc._jvm.org.apache.hadoop.fs.FileSystem
fs = FileSystem.get(URI(HDFS_NAMENODE), sc._jsc.hadoopConfiguration())

def hdfs_size_mb(path):
    p = Path(path)
    if not fs.exists(p):
        return None
    return fs.getContentSummary(p).getLength() / 1024 / 1024

def hdfs_ls(path, max_items=5):
    p = Path(path)
    if not fs.exists(p):
        return []
    return [(f.getPath().getName(), f.getLen()) for f in fs.listStatus(p)[:max_items]]

def ingest(name, csv_path, out_path):
    print(f"\n========= {name} =========")
    print(f"  src: {csv_path}")
    print(f"  dst: {out_path}")
    t0 = time.time()
    df = (
        spark.read
        .option("header", True)
        .option("encoding", "UTF-8")
        .option("multiLine", True)
        .option("quote", '"')
        .option("escape", '"')
        .csv(f"hdfs://10.43.97.164:9000{csv_path}")
    )
    cols = len(df.columns)
    print(f"  columns      : {cols}")
    # Sobrescribimos siempre — bronze parquet es idempotente
    (df.write
        .mode("overwrite")
        .option("compression", "snappy")
        .parquet(f"hdfs://10.43.97.164:9000{out_path}"))
    dur = time.time() - t0
    df_pq = spark.read.parquet(f"hdfs://10.43.97.164:9000{out_path}")
    n = df_pq.count()
    src_mb = hdfs_size_mb(csv_path)
    dst_mb = hdfs_size_mb(out_path)
    ratio = (src_mb or 0) / (dst_mb or 1)
    print(f"  rows escritas: {n:,}")
    print(f"  CSV    size  : {src_mb:>9.1f} MB")
    print(f"  Parquet size : {dst_mb:>9.1f} MB  ({ratio:.1f}x menor)")
    print(f"  tiempo total : {dur:.1f}s")
    return {"name": name, "rows": n, "csv_mb": src_mb, "pq_mb": dst_mb, "secs": dur}

## 2. MEN (5 MB, calentamiento)

In [3]:
res_men = ingest("MEN", P.BRONZE_CSV_MEN, P.BRONZE_PQ_MEN)


========= MEN =========
  src: /usr/proyecto/bronze/men/MEN_Estadisticas_Educacion_Municipio_20260519.csv
  dst: /usr/proyecto/bronze_parquet/men


  columns      : 41


26/05/19 16:32:26 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


  rows escritas: 15,707
  CSV    size  :       5.0 MB
  Parquet size :       1.3 MB  (3.8x menor)
  tiempo total : 12.3s


## 3. Internet Fijo (385 MB)

In [4]:
res_internet = ingest("INTERNET", P.BRONZE_CSV_INTERNET, P.BRONZE_PQ_INTERNET)


========= INTERNET =========
  src: /usr/proyecto/bronze/internet/Internet_Fijo_Accesos_por_tecnologia_y_segmento_20260407.csv
  dst: /usr/proyecto/bronze_parquet/internet


  columns      : 12


  rows escritas: 2,795,052
  CSV    size  :     384.3 MB
  Parquet size :      29.9 MB  (12.8x menor)
  tiempo total : 27.0s


## 4. SISBEN Personas (937 MB, 4.5M filas)

In [5]:
res_sisben = ingest("SISBEN", P.BRONZE_CSV_SISBEN, P.BRONZE_PQ_SISBEN)


========= SISBEN =========
  src: /usr/proyecto/bronze/sisben/DNP_Sisben_Personas_20260519.csv
  dst: /usr/proyecto/bronze_parquet/sisben


  columns      : 48


  rows escritas: 4,465,955
  CSV    size  :     937.1 MB
  Parquet size :      63.0 MB  (14.9x menor)
  tiempo total : 65.1s


## 5. ICFES Saber 11 (3.5 GB, 7.1M filas) — el grande

In [6]:
res_icfes = ingest("ICFES", P.BRONZE_CSV_ICFES, P.BRONZE_PQ_ICFES)


========= ICFES =========
  src: /usr/proyecto/bronze/icfes/Resultados_unicos_Saber_11_20260519.csv
  dst: /usr/proyecto/bronze_parquet/icfes
  columns      : 51


  rows escritas: 7,109,704
  CSV    size  :    3514.5 MB
  Parquet size :     292.0 MB  (12.0x menor)
  tiempo total : 159.2s


## 6. Resumen consolidado

In [7]:
import pandas as pd
resumen = pd.DataFrame([res_men, res_internet, res_sisben, res_icfes])
resumen["ratio_compresion"] = (resumen["csv_mb"] / resumen["pq_mb"]).round(2)
resumen[["name","rows","csv_mb","pq_mb","ratio_compresion","secs"]]

,name,rows,csv_mb,pq_mb,ratio_compresion,secs
0,MEN,15707,4.974689,1.302516,3.82,12.263437
1,INTERNET,2795052,384.285120,29.937064,12.84,26.962513
2,SISBEN,4465955,937.061273,62.967535,14.88,65.121461
3,ICFES,7109704,3514.540033,291.978954,12.04,159.189850


## 7. Inventario Bronze Parquet en HDFS

In [8]:
for d in [P.BRONZE_PQ_MEN, P.BRONZE_PQ_INTERNET, P.BRONZE_PQ_SISBEN, P.BRONZE_PQ_ICFES]:
    print(f"\n=== {d} ===")
    items = hdfs_ls(d, max_items=20)
    print(f"  archivos: {len(items)}")
    total = sum(sz for _, sz in items)
    print(f"  total: {total/1e6:.1f} MB")
    for name, sz in items[:5]:
        print(f"    - {name:<60s} {sz/1e6:>7.2f} MB")


=== /usr/proyecto/bronze_parquet/men ===
  archivos: 2
  total: 1.4 MB
    - _SUCCESS                                                        0.00 MB
    - part-00000-a20fb276-8534-4e5a-8648-3b25a1040483-c000.snappy.parquet    1.37 MB

=== /usr/proyecto/bronze_parquet/internet ===
  archivos: 2
  total: 31.4 MB
    - _SUCCESS                                                        0.00 MB
    - part-00000-7c14db27-b62b-449c-99df-07f927466f1f-c000.snappy.parquet   31.39 MB

=== /usr/proyecto/bronze_parquet/sisben ===
  archivos: 2
  total: 66.0 MB
    - _SUCCESS                                                        0.00 MB
    - part-00000-9c1faaf0-2f42-4d3f-80fd-425e2a4290df-c000.snappy.parquet   66.03 MB

=== /usr/proyecto/bronze_parquet/icfes ===
  archivos: 2
  total: 306.2 MB
    - _SUCCESS                                                        0.00 MB
    - part-00000-a195ce89-3f66-40d6-a975-e28600fd56b1-c000.snappy.parquet  306.16 MB


## 8. Conclusión

Si el resumen muestra row counts > 0 para los 4 datasets y un ratio de compresión sano (~3-7×),
Bronze está listo.

**Siguiente paso:** `02_silver_icfes.ipynb` — limpieza, normalización y tipado de ICFES.

In [9]:
spark.stop()